# Reconhecimento de Gestos com Webcam, OpenCV e MediaPipe

Este notebook utiliza a sua webcam para reconhecer gestos das mãos em tempo real. Ele usa a biblioteca **OpenCV** para capturar os vídeos da câmera e o **MediaPipe Tasks** para processar a imagem e identificar os gestos.

In [ ]:
!pip install opencv-python mediapipe numpy urllib3

### Baixar o modelo de Reconhecimento de Gestos do MediaPipe
O MediaPipe Gesture Recognizer requer um arquivo de modelo (`.task`). Vamos baixar o modelo padrão que reconhece gestos como polegar para cima, vitória, etc.

In [ ]:
import urllib.request
import os

model_path = 'gesture_recognizer.task'
url = 'https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task'

if not os.path.exists(model_path):
    print("Baixando o modelo...")
    urllib.request.urlretrieve(url, model_path)
    print("Modelo baixado com sucesso!")
else:
    print("O modelo já existe na pasta.")

### Reconhecimento de Gestos com a Webcam
**Atenção:** Ao rodar a célula abaixo, será aberta uma **nova janela** no seu sistema que acessará a sua câmera.

Para fechar a janela, clique nela de forma a ativá-la e aperte a tecla **`q`** ou **`ESC`** no seu teclado. O código irá desenhar os pontos das mãos e o nome do gesto detectado.

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np

# 1. Instanciar o reconhecedor de gestos
base_options = python.BaseOptions(model_asset_path='gesture_recognizer.task')
options = vision.GestureRecognizerOptions(base_options=base_options, min_hand_detection_confidence=0.5)
recognizer = vision.GestureRecognizer.create_from_options(options)

# Ferramentas de desenho do MediaPipe
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
mp_drawing_styles = mp.solutions.drawing_styles

def draw_landmarks_and_gestures(image, recognition_result):
    """
    Desenha os pontos da mão e o nome do gesto detectado na imagem.
    """
    if not recognition_result.hand_landmarks:
        return image

    for i, hand_landmarks in enumerate(recognition_result.hand_landmarks):
        # Converter landmarks para o formato protobuffer que o draw_landmarks espera
        hand_landmarks_proto = mp.framework.formats.landmark_pb2.NormalizedLandmarkList()
        hand_landmarks_proto.landmark.extend([
            mp.framework.formats.landmark_pb2.NormalizedLandmark(x=landmark.x, y=landmark.y, z=landmark.z) 
            for landmark in hand_landmarks
        ])
        
        # Desenhar os landmarks e conexões
        mp_drawing.draw_landmarks(
            image,
            hand_landmarks_proto,
            mp_hands.HAND_CONNECTIONS,
            mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())

        # Pegar o gesto detectado para esta mão
        if recognition_result.gestures and len(recognition_result.gestures) > i:
            gesture = recognition_result.gestures[i][0]
            gesture_name = gesture.category_name
            score = round(gesture.score, 2)
            
            # Posicionar o texto próximo ao pulso (ponto 0)
            wrist = hand_landmarks[0]
            h, w, _ = image.shape
            text_pos = (int(wrist.x * w), int(wrist.y * h) - 20)
            
            cv2.putText(image, f"{gesture_name} ({score})", text_pos, 
                        cv2.FONT_HERSHEY_DUPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA)
            
    return image

# 2. Abrir a Webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Não foi possível abrir a câmera.")
else:
    try:
        print("Câmera aberta. Pressione 'q' na janela de vídeo para sair.")
        while cap.isOpened():
            success, image = cap.read()
            if not success:
                break

            # Espelhar imagem horizontalmente para facilitar a interação
            image = cv2.flip(image, 1)
            
            # Converter BGR para RGB (MediaPipe usa RGB)
            rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

            # Reconhecer o gesto
            recognition_result = recognizer.recognize(mp_image)

            # Desenhar resultados (pontos e gestos)
            annotated_image = draw_landmarks_and_gestures(image, recognition_result)

            # Mostrar o resultado
            cv2.imshow('MediaPipe Reconhecimento de Gestos', annotated_image)

            # Parar se apertarem 'q' ou 'ESC'
            key = cv2.waitKey(5) & 0xFF
            if key == ord('q') or key == 27:
                print("Encerrando câmera...")
                break
    except Exception as e:
        print(f"Ocorreu um erro: {e}")
    finally:
        cap.release()
        cv2.destroyAllWindows()
